<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

# 3. Simulation

An Allo kernel is just a Python callable: calling it *runs* it. This notebook is about **running and validating kernels functionally** — getting numbers out and checking them against a reference — before you ever worry about clock cycles or resource usage. We cover the two backends that execute a kernel (the **CPU** JIT path and **Vitis C-simulation**), the calling convention that moves NumPy buffers and scalars across the Python boundary, how local dataflow `Stream`s are simulated, and how results are cached so the edit–run loop stays fast.

This is notebook 3 of a 4-part series: **1. Frontend** (writing kernels — the *what*), **2. Scheduling** (transforming kernels — the *how*), **3. Simulation** (this notebook — running and validating), **4. Backend** (synthesis, resource and timing reports, and hardware). Synthesis is only previewed here; notebook 4 drives it in full.

In [ ]:
import os
import numpy as np
import tempfile
from allo.lang import kernel, Stream, range, i32, f32, apint
from allo.backend.vitis.utils import is_vitis_available
print("Vitis toolchain available:", is_vitis_available())

## 1. A kernel is a callable

The simplest way to run a kernel is to **call it**. A direct call transparently compiles the kernel with the **CPU backend** using default options, runs it, and writes any output buffers back in place. Because a kernel behaves like an ordinary Python function, you can freely interleave NumPy, plain Python helpers, and kernel calls in the same program.

There are two ways to reach a backend:

- **Direct call** — `vec_add(A, B, C)` — runs on the CPU backend with default options. Zero ceremony; ideal for a quick functional check.
- **`kernel.schedule().export(backend, **kwargs)`** — applies any schedule (see notebook 2), binds the result back to the kernel, and returns a *backend object* you call or drive explicitly. This is the path for a transformed kernel or for the Vitis flow.

In [ ]:
N = 64

@kernel
def vec_add(A: f32[N], B: f32[N], C: f32[N]):
    for i in range(N):
        C[i] = A[i] + B[i]

A = np.arange(N, dtype=np.float32)
B = np.arange(N, dtype=np.float32) * 10.0
C = np.zeros(N, dtype=np.float32)

vec_add(A, B, C)                 # CPU backend, default options; C written in place
np.testing.assert_allclose(C, A + B)

# Kernel calls mix freely with NumPy: use the result like any array.
D = np.sqrt(C) + 1.0
print("C[:5]  =", C[:5])
print("mixed  =", D[:5])
print("section 1 OK")

## 2. The CPU backend, explicitly

For control over CPU options or to run a **scheduled** kernel, export a backend object instead of calling the kernel directly:

```python
backend = kernel.schedule().export("cpu", opt_level=3)
```

`opt_level` is the LLVM optimization level (default `2`). Calling the backend is the same as `.run(...)`: `backend(A, B, C)` is equivalent to `backend.run(A, B, C)`. The CPU backend executes the **same numeric semantics** the hardware flow will — including local dataflow streams — so it is a convenient reference for expected behavior.

**Caveat.** The CPU path is for *functional* validation only. It is not fully verified and, like real hardware, an imbalanced stream producer/consumer pair **can deadlock**. When you need an authoritative functional check, prefer **Vitis C-simulation** (section 4): it compiles and runs the real HLS C++ and needs no synthesis.

In [ ]:
@kernel
def scale_add(A: f32[N], B: f32[N], C: f32[N]):
    for i in range(N):
        C[i] = A[i] * 2.0 + B[i]

s = scale_add.schedule()
s.pipeline(s.loop("i"), ii=1)          # a schedule transform (see notebook 2)
backend = s.export("cpu", opt_level=3)

C1 = np.zeros(N, dtype=np.float32)
C2 = np.zeros(N, dtype=np.float32)
backend(A, B, C1)                      # calling the backend ...
backend.run(A, B, C2)                  # ... is the same as .run()

ref = A * 2.0 + B
np.testing.assert_allclose(C1, ref)
np.testing.assert_allclose(C2, ref)
print("backend(...) == backend.run(...):", np.array_equal(C1, C2))
print("section 2 OK")

## 3. The calling convention

**Buffer arguments** are NumPy arrays. The backend validates each array's shape and dtype against the kernel annotation and writes results **back in place**. The recommended style is an explicit *output buffer* you allocate and pass in — it behaves identically on CPU and Vitis.

**Scalar arguments** are plain Python numbers, validated against their annotation. **Scalar return values** work on both backends.

Two boundary rules to remember:

- On the **Vitis** top kernel, **shaped return values are rejected** — a kernel that must produce an array has to write it into a buffer argument. (Scalar returns are fine everywhere.) This is another reason to prefer output buffers for portability.
- A **non-standard `apint` width** (say `apint(5)`) is widened to the next standard width — 8/16/32/64 bits — at the host boundary, matching the `generate-apint-wrapper` ABI. The *arithmetic inside the kernel still wraps at the declared width*; only the way the value is passed in and out is widened.

In [ ]:
# Buffer output written in place, then read back:
@kernel
def scale(x: f32[8], out: f32[8]):
    for i in range(8):
        out[i] = x[i] * 3.0

x = np.arange(8, dtype=np.float32)
out = np.zeros(8, dtype=np.float32)
scale(x, out)                          # `out` is filled by the call
np.testing.assert_allclose(out, x * 3.0)

# Scalar return value (works on CPU and Vitis):
@kernel
def reduce_sum(A: i32[6]) -> i32:
    s: i32 = 0
    for i in range(6):
        s = s + A[i]
    return s

total = reduce_sum(np.array([1, 2, 3, 4, 5, 6], dtype=np.int32))
assert int(total) == 21
print("out   =", out[:4], "...")
print("total =", int(total))
print("section 3a OK")

In [ ]:
# apint boundary widening: i5/u5 are passed as int8 at the boundary,
# but arithmetic wraps modulo 2**5 inside the kernel.
i5 = apint(5, signed=True)
u5 = apint(5, signed=False)

@kernel
def addsub(A: i5[8], B: u5[8], C: i5[8]):
    for i in range(8):
        C[i] = A[i] + B[i]

A5 = np.array([-4, -3, -2, -1, 0, 1, 2, 3], dtype=np.int8)   # host width = int8
B5 = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.uint8)
C5 = np.zeros(8, dtype=np.int8)
addsub(A5, B5, C5)

# 5-bit signed wraparound: (a + b) mod 32 mapped back into [-16, 15]
expected = ((A5.astype(np.int16) + B5 + 16) % 32 - 16).astype(np.int8)
np.testing.assert_array_equal(C5, expected)
print("i5 result (wraps mod 32):", C5)
print("section 3b OK")

## 4. Vitis C-simulation: the reliable functional check

Vitis **C-simulation (csim)** compiles the *actual* generated HLS C++ into a shared library and calls the top function from Python — no host program, no synthesis, just real HLS semantics behind the same NumPy calling convention. It is the authoritative functional check.

Two useful entry points:

- `backend.hls_code` returns the generated synthesizable C++ as a string. It needs **no toolchain**, so it is the fastest way to inspect codegen.
- Calling the backend — `backend(A, B, C)`, equivalently `backend.csim(A, B, C)` — builds `libkernel.so` and runs the top function.

Every real-Vitis cell below is gated on `is_vitis_available()` so the notebook still passes without a toolchain. **In this environment Vitis is available**, so they execute.

Rule of thumb: **CPU sim** = fast JIT reference; **Vitis csim** = real HLS C++, authoritative. Cross-check with csim whenever correctness matters.

In [ ]:
code = vec_add.schedule().export("vitis").hls_code
assert "void vec_add(" in code

# Show the top-function signature and the first lines of the body:
start = code.index("void vec_add(")
print(code[start:start + 300])
print("...")
print("section 4a OK  (hls_code needs no toolchain)")

In [ ]:
if is_vitis_available():
    with tempfile.TemporaryDirectory() as proj:
        vb = vec_add.schedule().export("vitis", project_path=proj)
        Cv = np.zeros(N, dtype=np.float32)
        vb(A, B, Cv)                       # csim == vb.csim(A, B, Cv)
    np.testing.assert_allclose(Cv, A + B, rtol=1e-5)
    print("Vitis csim matches NumPy:", bool(np.allclose(Cv, A + B)))
else:
    print("Vitis not available — skipping csim")
print("section 4b OK")

## 5. Dataflow stream simulation

Kernels talk to nested kernels through local `Stream` FIFOs. The CPU backend simulates these with a small dataflow runtime — there is no separate API, just call the top kernel. When two or more **contiguous** nested-kernel calls are wired together by stream arguments, the lowering runs them **concurrently** (OpenMP sections), so a producer and consumer overlap the way pipelined hardware stages do. Each lane is a **bounded FIFO**: `put` blocks when the lane is full and `get` blocks when it is empty — matching hardware, and meaning an imbalanced pair can **deadlock**.

Payload shapes:

- **Scalar** payload — `Stream[i32]` — one value per `put`/`get`.
- **Block** payload — `Stream[i32[2, 2]]` — a whole contiguous block per transfer.
- **Stream array** — `Stream[i32][2, 2]` — several independent FIFO lanes selected by index (used for PE arrays; see notebook 1).

Current restrictions of the dataflow simulator are intentionally simple:

- Stream-connected calls in a dataflow group must be **contiguous**.
- A group cannot **mix** stream-connected and non-stream calls.
- Stream-connected calls must **not return values** — pass output buffers instead.

In [ ]:
@kernel
def stream_top(x: i32[8], out: i32[8]):
    fifo: Stream[i32]

    @kernel
    def producer(src: i32[8], stream: Stream[i32]):
        for i in range(8):
            stream.put(src[i] + 1)

    @kernel
    def consumer(stream: Stream[i32], dst: i32[8]):
        for i in range(8):
            dst[i] = stream.get() * 2

    producer(x, fifo)
    consumer(fifo, out)

xs = np.arange(8, dtype=np.int32)
outs = np.zeros(8, dtype=np.int32)
stream_top(xs, outs)                    # producer/consumer run concurrently
np.testing.assert_array_equal(outs, (xs + 1) * 2)
print("scalar stream:", outs)
print("section 5a OK")

In [ ]:
@kernel
def block_top(out: i32[2, 2, 2]):
    fifo: Stream[i32[2, 2]]

    @kernel
    def producer(stream: Stream[i32[2, 2]]):
        buf: i32[2, 2]
        buf[0, 0] = 1
        buf[0, 1] = 2
        buf[1, 0] = 3
        buf[1, 1] = 4
        stream.put(buf)
        buf[0, 0] = 10
        buf[0, 1] = 20
        buf[1, 0] = 30
        buf[1, 1] = 40
        stream.put(buf)

    @kernel
    def consumer(stream: Stream[i32[2, 2]], dst: i32[2, 2, 2]):
        first = stream.get()
        second = stream.get()
        dst[0, 0, 0] = first[0, 0]
        dst[0, 0, 1] = first[0, 1]
        dst[0, 1, 0] = first[1, 0]
        dst[0, 1, 1] = first[1, 1]
        dst[1, 0, 0] = second[0, 0]
        dst[1, 0, 1] = second[0, 1]
        dst[1, 1, 0] = second[1, 0]
        dst[1, 1, 1] = second[1, 1]

    producer(fifo)
    consumer(fifo, out)

blk = np.zeros((2, 2, 2), dtype=np.int32)
block_top(blk)                          # each put/get moves a whole 2x2 block
expected = np.array([[[1, 2], [3, 4]], [[10, 20], [30, 40]]], dtype=np.int32)
np.testing.assert_array_equal(blk, expected)
print("block stream:\n", blk)
print("section 5b OK")

## 6. Bit-manipulation simulation

Bit slicing and single-bit access lower to real hardware bit ops, and the simulator reproduces them exactly — which makes pack/unpack and bit-reversal great round-trip tests. `x[lo:hi]` reads/writes a half-open bit range (constant width, possibly dynamic offset) and `x[k]` reads/writes one bit.

In [ ]:
u32 = apint(32, signed=False)
u8 = apint(8, signed=False)

# Unpack four bytes from each packed 32-bit word (dynamic offset, width-8 slices):
@kernel
def unpack(packed: u32[6], out: i32[6, 4]):
    for i in range(6):
        for p in range(4):
            out[i, p] = packed[i][p * 8 : p * 8 + 8]

lanes = np.random.randint(0, 256, (6, 4)).astype(np.uint32)
# NumPy reference (vectorized): pack lanes into words, expect to recover `lanes`.
shifts = (8 * np.arange(4)).astype(np.uint32)
packed = (lanes << shifts).sum(axis=1).astype(np.uint32)

unpacked = np.zeros((6, 4), dtype=np.int32)
unpack(packed, unpacked)
np.testing.assert_array_equal(unpacked, lanes.astype(np.int32))

# Reverse the 8 bits of each byte (width-1 slice writes):
@kernel
def rev(src: u8[8], out: u8[8]):
    for i in range(8):
        r: u8 = 0
        for b in range(8):
            r[7 - b] = src[i][b]
        out[i] = r

src = np.random.randint(0, 256, 8).astype(np.uint8)
rout = np.zeros(8, dtype=np.uint8)
rev(src, rout)
ref_rev = np.array([int(f"{int(v):08b}"[::-1], 2) for v in src], dtype=np.uint8)
np.testing.assert_array_equal(rout, ref_rev)
print("unpack round-trip OK; bit-reverse OK")
print("section 6 OK")

## 7. Vitis run modes

`backend.run(mode, *args)` is the single dispatch point for the Vitis flow:

| Mode       | Needs a platform? | What it does                                          |
| ---------- | ----------------- | ----------------------------------------------------- |
| `"csim"`   | no                | Python-native C-simulation (same as calling backend). |
| `"csyn"`   | no                | HLS C-to-RTL synthesis (== `synth()`; no run args).   |
| `"hw_emu"` | **yes**           | Hardware emulation through `v++`/XRT.                  |
| `"hw"`     | **yes**           | Full hardware build and on-board run.                 |
| `"sw_emu"` | no                | Deprecated alias; runs `csim` and warns.              |

`csim`/`csyn` need no platform. `hw_emu`/`hw` build through `v++` and run an XRT host, so they need a platform exported (`export PLATFORM=/path/<shell>.xpfm`). `backend.precheck(mode)` scaffolds and validates a buildable project **without** the long, platform-locked link step. All run/csim/synth methods accept `exist_ok=True` (the default); pass `exist_ok=False` to force a rebuild.

Here we only exercise `csim` — synthesis, emulation, and hardware are notebook 4.

In [ ]:
if is_vitis_available():
    with tempfile.TemporaryDirectory() as proj:
        backend = vec_add.schedule().export("vitis", project_path=proj)
        Cc = np.zeros(N, dtype=np.float32)
        backend.run("csim", A, B, Cc)          # explicit csim via run(mode, ...)
    np.testing.assert_allclose(Cc, A + B, rtol=1e-5)
    print('backend.run("csim", ...) matches NumPy:', bool(np.allclose(Cc, A + B)))
else:
    print("Vitis not available — skipping csim")
print("section 7 OK")

## 8. Caching

Repeated runs stay cheap because both backends cache.

- **CPU** uses an **in-process compile cache** keyed on the kernel IR and the CPU configuration (opt level, shared libs). A repeated call with the same kernel and config reuses the already-built MLIR execution engine — no re-lowering.
- **Vitis** uses an in-process cache *and* a **disk cache** for csim projects, under `$HOME/.allo/cache/vitis/csim/<key>/` (holding `kernel.cpp`, `kernel.h`, `csim.mk`, and a `cache.json`). When the cached build exists, it is reused.

Pass `exist_ok=False` to a run/csim/synth call to force a rebuild.

The CPU cell below times a fresh kernel's first call (which compiles) against a second identical call (which reuses the engine).

In [ ]:
import time

@kernel
def cached_add(A: f32[N], B: f32[N], C: f32[N]):
    for i in range(N):
        C[i] = A[i] + B[i] + 1.0

Cc = np.zeros(N, dtype=np.float32)

t0 = time.perf_counter(); cached_add(A, B, Cc); first = time.perf_counter() - t0
t0 = time.perf_counter(); cached_add(A, B, Cc); second = time.perf_counter() - t0
np.testing.assert_allclose(Cc, A + B + 1.0)

print(f"first call : {first * 1e3:8.3f} ms  (compile + run)")
print(f"second call: {second * 1e3:8.3f} ms  (cached engine reused)")
print("section 8a OK")

In [ ]:
csim_cache = os.path.expanduser("~/.allo/cache/vitis/csim")
if is_vitis_available():
    with tempfile.TemporaryDirectory() as proj:
        backend = vec_add.schedule().export("vitis", project_path=proj)
        Cv = np.zeros(N, dtype=np.float32)
        t0 = time.perf_counter()
        backend(A, B, Cv)                  # vec_add was csim'd earlier -> cache hit
        elapsed = time.perf_counter() - t0
    np.testing.assert_allclose(Cv, A + B, rtol=1e-5)
    print(f"vec_add csim (cache hit expected): {elapsed:.3f} s")
    if os.path.isdir(csim_cache):
        keys = sorted(os.listdir(csim_cache))
        print(f"disk cache holds {len(keys)} project(s) under {csim_cache}")
        if keys:
            print("  one entry contains:", sorted(os.listdir(os.path.join(csim_cache, keys[0]))))
else:
    print("Vitis not available — skipping Vitis cache demo")
print("section 8b OK")

## 9. A recommended validation workflow

Putting it together, the development loop that scales from a toy kernel to a real design is:

1. **Write** the kernel with in-place output buffers.
2. Compute a **NumPy reference**.
3. Assert on the **CPU backend** (`np.testing.assert_allclose`) for a fast check.
4. **Confirm on Vitis csim** (gated on availability) for the authoritative check.

Here is that loop on a small buffer-mode GEMM — a template you can copy for your own kernels.

In [ ]:
M, K, P = 4, 5, 6

@kernel
def gemm(Amat: f32[M, K], Bmat: f32[K, P], Cmat: f32[M, P]):
    for i in range(M):
        for j in range(P):
            acc: f32 = 0.0
            for k in range(K):
                acc = acc + Amat[i, k] * Bmat[k, j]
            Cmat[i, j] = acc

Am = np.random.rand(M, K).astype(np.float32)
Bm = np.random.rand(K, P).astype(np.float32)
gemm_ref = Am @ Bm

# 3. CPU functional check (fast)
Ccpu = np.zeros((M, P), dtype=np.float32)
gemm(Am, Bm, Ccpu)
np.testing.assert_allclose(Ccpu, gemm_ref, rtol=1e-5, atol=1e-5)
print("CPU GEMM matches NumPy")

# 4. Authoritative Vitis csim check
if is_vitis_available():
    with tempfile.TemporaryDirectory() as proj:
        gb = gemm.schedule().export("vitis", project_path=proj)
        Chls = np.zeros((M, P), dtype=np.float32)
        gb(Am, Bm, Chls)
    np.testing.assert_allclose(Chls, gemm_ref, rtol=1e-4, atol=1e-4)
    print("Vitis csim GEMM matches NumPy")
else:
    print("Vitis not available — CPU check only")
print("section 9 OK")